# Train & evaluate the MoE toxicity classifier on Colab

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/qpp112/toxic-comment-moe/blob/main/notebooks/train_on_colab.ipynb)

This notebook reproduces every number in the [README](https://github.com/qpp112/toxic-comment-moe):

1. checks the GPU and installs the package,
2. runs the CPU test suite (1 min, catches problems before any GPU time is spent),
3. downloads the Jigsaw data with your Kaggle API token,
4. trains the 8-run **loss × architecture ablation** on the full dataset,
5. builds the results tables and figures,
6. lets you try the trained MoE model on your own sentences.

**Before you start:** *Runtime → Change runtime type → T4 GPU* (or better).

**Time:** roughly 15–20 min per run on a free T4, so about 2–3 h for all 8 runs (an A100 is
~4× faster). Results are written to Google Drive and finished runs are skipped. **If Colab
disconnects, just run all cells again** and it picks up where it stopped.

In [ ]:
import torch

print("torch", torch.__version__, "| CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("No GPU found. Runtime -> Change runtime type -> T4 GPU, then re-run this cell.")

## 1. Settings, Google Drive, install

In [ ]:
from pathlib import Path

REPO_URL = "https://github.com/qpp112/toxic-comment-moe.git"
USE_DRIVE = True  # keep data, tokenization cache and results on Drive so a disconnect loses nothing

if USE_DRIVE:
    from google.colab import drive

    drive.mount("/content/drive")
    WORK = Path("/content/drive/MyDrive/toxic-moe")
else:
    WORK = Path("/content/toxic-moe-work")

DATA_DIR = WORK / "data"
RESULTS_DIR = WORK / "results"
for d in (DATA_DIR, RESULTS_DIR):
    d.mkdir(parents=True, exist_ok=True)
print("data   ->", DATA_DIR)
print("results->", RESULTS_DIR)

In [ ]:
%cd /content
!test -d toxic-comment-moe || git clone -q {REPO_URL}
%cd /content/toxic-comment-moe
!git pull -q
!pip install -q -e ".[dev]"

## 2. Run the test suite (CPU, ~1 min)

In [ ]:
!python -m pytest -q

## 3. Get the data

You need a Kaggle account. One-time setup:

1. Accept the competition rules: <https://www.kaggle.com/competitions/jigsaw-toxic-comment-classification-challenge/rules>
2. Kaggle → *Settings* → *API* → **Create New Token**, which downloads `kaggle.json`.
3. Either add `KAGGLE_USERNAME` and `KAGGLE_KEY` as Colab secrets (🔑 icon in the left bar),
   or upload `kaggle.json` when the cell below asks for it.

Alternatively, put `train.csv`, `test.csv` and `test_labels.csv` in `MyDrive/toxic-moe/data/` yourself.

In [ ]:
import os

needed = ["train.csv", "test.csv", "test_labels.csv"]
if all((DATA_DIR / f).exists() for f in needed):
    print("Data already present:", sorted(p.name for p in DATA_DIR.glob("*.csv")))
else:
    kaggle_json = Path.home() / ".kaggle" / "kaggle.json"
    if not kaggle_json.exists():
        try:
            from google.colab import userdata

            os.environ["KAGGLE_USERNAME"] = userdata.get("KAGGLE_USERNAME")
            os.environ["KAGGLE_KEY"] = userdata.get("KAGGLE_KEY")
            print("Using Kaggle credentials from Colab secrets")
        except Exception:
            from google.colab import files

            print("Upload kaggle.json")
            uploaded = files.upload()
            kaggle_json.parent.mkdir(exist_ok=True)
            kaggle_json.write_bytes(next(iter(uploaded.values())))
            kaggle_json.chmod(0o600)
    !pip install -q kaggle
    !bash scripts/download_data.sh {DATA_DIR}

## 4. Quick pipeline check (~5 min)

One MoE run on 2% of the data for one epoch. This also tokenizes and caches the 64k-comment
test set on Drive, so later runs start faster. The scores here are meaningless; it only checks that everything runs end to end.

In [ ]:
!python scripts/run_ablation.py --only moe_cbfocal --set train_fraction=0.02 epochs=1 data_dir={DATA_DIR} output_dir=/content/smoke_results --force

## 5. Full ablation (the long one)

Runs in the order listed in `configs/ablation.yaml`; the four most informative runs come first.
To do only the headline comparison, add `--only bert_bce moe_cbfocal`.

In [ ]:
!python scripts/run_ablation.py --list --set data_dir={DATA_DIR} output_dir={RESULTS_DIR}

In [ ]:
!python scripts/run_ablation.py --set data_dir={DATA_DIR} output_dir={RESULTS_DIR}

## 6. Report

In [ ]:
import re

from IPython.display import Image, Markdown, display

!python scripts/make_report.py --results {RESULTS_DIR} --update-readme

summary = (RESULTS_DIR / "summary.md").read_text()
display(Markdown(re.sub(r"<picture>.*?</picture>", "", summary, flags=re.S)))
for stem in ["ablation", "per_label_f1", "pr_curves", "expert_routing"]:
    path = Path("docs/figures") / f"{stem}_light.png"
    if path.exists():
        display(Image(str(path)))

### Save the results

Download a small zip with the metrics, configs, figures and the updated README. It is a few MB
and excludes checkpoints and predictions. Commit its contents to the repository.

In [ ]:
import zipfile

bundle = Path("/content/toxic_moe_results.zip")
with zipfile.ZipFile(bundle, "w", zipfile.ZIP_DEFLATED) as z:
    for p in RESULTS_DIR.rglob("*"):
        if p.is_file() and p.suffix in {".json", ".yaml", ".md"}:
            z.write(p, Path("results") / p.relative_to(RESULTS_DIR))
    for p in Path("docs/figures").glob("*.png"):
        z.write(p, p)
    z.write("README.md", "README.md")
print(f"{bundle} ({bundle.stat().st_size / 1e6:.1f} MB)")

from google.colab import files

files.download(str(bundle))

## 7. Try the model

Loads the saved `moe_cbfocal` checkpoint (the only run that keeps its weights) and scores some
sentences. `flagged` uses the per-label thresholds tuned on the validation split. The last
columns show how the router split each comment across the six experts.

In [ ]:
import sys

sys.path.insert(0, "/content/toxic-comment-moe/src")  # editable installs need a restart otherwise

import pandas as pd

from toxic_moe.predict import ToxicityPredictor

predictor = ToxicityPredictor(RESULTS_DIR / "moe_cbfocal")

examples = [
    "Thanks for fixing the citation, much appreciated!",
    "Please stop reverting my edits without explaining why on the talk page.",
    "You are an idiot and nobody wants you editing here.",
    "If you touch this article again I will find you and make you regret it.",
    "People from that country are all criminals and should be banned.",
    "This is the dumbest, most useless garbage I have ever read.",
]
rows = predictor(examples)
table = pd.DataFrame(
    [
        {
            "text": r["text"][:70],
            **r["probabilities"],
            "flagged": ", ".join(r["flagged"]) or "-",
            **{f"E{i + 1}": w for i, w in enumerate(r["expert_weights"])},
        }
        for r in rows
    ]
)
table.style.format(precision=2).background_gradient(cmap="Blues", subset=list(rows[0]["probabilities"]))

In [ ]:
# Your own sentence:
text = "Type something here"
predictor([text])[0]